# Part 2 — XGBoost Tweedie (Emerging Market)

Fits Tweedie XGBoost (power=1.5) for motor, buildings, and contents using data-driven feature selection.

**Key adaptations vs v4:**
- k-fold CV (5 folds) instead of time-based split (only 2 years available)
- Simpler model: max_depth=4, n_estimators=200, heavier regularization (5% of data vs developed)
- Data-driven feature selection (no Features_v3.xlsx — auto-select all useful features)
- Age band mapped to numeric midpoints (done in Notebook 01)
- Trend factors from Part 1 applied to 2025 test predictions

In [0]:
# Step 1: Load processed training data, test data, and enrichment files
import pandas as pd
import numpy as np
import os

%pip install xgboost openpyxl -q

DATA_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/data_emerging/'
PROC_DIR = os.path.join(DATA_DIR, 'proc')

# Load processed training data (from Notebook 1)
train = {}
for line in ['motor', 'buildings', 'contents']:
    train[line] = pd.read_parquet(os.path.join(PROC_DIR, f'proc_{line}.parquet'))
    print(f'{line}: {train[line].shape} rows, target mean={train[line]["ultimate_claims"].mean():.4f}')

# Load test files
test = {}
for line in ['motor', 'buildings', 'contents']:
    test[line] = pd.read_parquet(os.path.join(DATA_DIR, f'test_{line}.parquet'))
    print(f'test_{line}: {test[line].shape}')

# Load enrichment files
customers = pd.read_parquet(os.path.join(DATA_DIR, 'customers.parquet'))
addresses = pd.read_parquet(os.path.join(DATA_DIR, 'addresses.parquet'))
weather = pd.read_parquet(os.path.join(DATA_DIR, 'weather.parquet'))
sample_submission = pd.read_parquet(os.path.join(DATA_DIR, 'sample_submission.parquet'))

# Trend factors for 2025 predictions (borrowed from Part 1)
TREND_FACTORS = {'motor': 1.04, 'buildings': 1.096, 'contents': 1.049}
print(f'\nTrend uplift factors for 2025: {TREND_FACTORS}')
print(f'sample_submission: {sample_submission.shape}')
print(f'Row ID prefixes: {sample_submission["row_id"].str[0].value_counts().to_dict()}')

In [0]:
# Step 2: Prepare test data (clean + join enrichment — same as Notebook 1)

AGE_BAND_MAP = {'17-24': 20.5, '25-34': 29.5, '35-49': 42.0, '50-64': 57.0, '65+': 70.0}

for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    # Fix exposure
    ps = pd.to_datetime(df['period_start'])
    pe = pd.to_datetime(df['period_end'])
    df['exposure'] = ((pe - ps).dt.days + 1) / 365.0
    # Create Year
    df['Year'] = ps.dt.year
    # Strip IDs
    for col in ['policy_id', 'customer_id', 'address_id', 'row_id']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    # Map age bands
    for col in df.columns:
        if col.endswith('_age_band') or col == 'age_band':
            new_col = col.replace('_age_band', '_age')
            df[new_col] = df[col].map(AGE_BAND_MAP)
    # Construction type
    if 'construction_type' in df.columns:
        df['construction_type'] = df['construction_type'].astype(str).str.strip().str.upper()
        df['construction_type'] = df['construction_type'].replace({'W': 'TIMBER', 'T': 'TIMBER'})
    # Drop compulsory_excess if constant
    if 'compulsory_excess' in df.columns and df['compulsory_excess'].nunique() <= 1:
        df.drop('compulsory_excess', axis=1, inplace=True)

# Zero-pad customer_id if needed (check lengths)
for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    if 'customer_id' in df.columns:
        lens = df['customer_id'].str.len().value_counts()
        if lens.nunique() > 1:
            max_len = lens.index.max()
            df['customer_id'] = df['customer_id'].str.zfill(max_len)

# Also check customers file
cust_lens = customers['customer_id'].str.len().value_counts()
if cust_lens.nunique() > 1:
    max_len = cust_lens.index.max()
    customers['customer_id'] = customers['customer_id'].str.zfill(max_len)

# Join enrichment to test files
customers_clean = customers.drop(columns=['surname', 'date_of_birth'], errors='ignore')
weather_agg = weather.groupby(['address_id', 'year'], as_index=False).mean(numeric_only=True)
weather_agg.rename(columns={'year': 'Year'}, inplace=True)

for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    df = df.merge(customers_clean, on='customer_id', how='left', suffixes=('', '_cust'))
    df = df.merge(addresses, on='address_id', how='left', suffixes=('', '_addr'))
    df = df.merge(weather_agg, on=['address_id', 'Year'], how='left', suffixes=('', '_weather'))
    test[line] = df
    print(f'test_{line}: {df.shape}')

In [0]:
# Step 3: Data-driven feature selection
# Auto-select all useful features, excluding targets, IDs, dates, PII, and leakage columns.

EXCLUDE_COLS = {
    # Targets
    'ultimate_claims', 'claim_count', 'reported_incurred', 'ultimate_no_dev',
    # IDs
    'row_id', 'policy_id', 'customer_id', 'address_id', 'claim_id',
    # Dates
    'period_start', 'period_end', 'loss_date', 'report_date',
    # PII
    'surname', 'date_of_birth', 'email_domain',
    # Leakage / metadata
    'written_premium', 'customer_service_contacts', 'renewal_decision',
    # Raw age band columns (already mapped to numeric _age)
    'driver_age_band', 'policyholder_age_band',
}

def select_features(df, exclude_set=EXCLUDE_COLS):
    """Select all columns except excluded ones, IDs, dates, and constants."""
    features = []
    for col in df.columns:
        if col in exclude_set:
            continue
        if col.startswith('_'):
            continue
        # Skip constant columns
        if df[col].nunique() <= 1:
            continue
        # Skip columns with all NaN
        if df[col].isna().all():
            continue
        features.append(col)
    return features

feature_selection = {}
for line in ['motor', 'buildings', 'contents']:
    feats = select_features(train[line])
    feature_selection[line] = feats
    print(f'{line}: {len(feats)} features selected')
    print(f'  {feats}')
    print()

In [0]:
# Step 4: Fit XGBoost Tweedie with k-fold CV
# Adaptations vs v4: k-fold CV (5 folds), simpler model (max_depth=4, 200 trees), heavier regularization

import xgboost as xgb
from sklearn.model_selection import KFold

def tweedie_deviance(y, y_pred, p=1.5):
    y = np.array(y, dtype=float)
    y_pred = np.maximum(y_pred, 1e-20)
    return 2 * (np.power(np.maximum(y, 0), 2-p) / ((1-p)*(2-p))
                - y * np.power(y_pred, 1-p) / (1-p)
                + np.power(y_pred, 2-p) / (2-p))

def tweedie_d2(y, y_pred, p=1.5):
    dev = tweedie_deviance(y, y_pred, p).sum()
    dev_null = tweedie_deviance(y, np.mean(y), p).sum()
    return 1 - dev / dev_null

def prepare_features(train_df, test_df, feature_list):
    """Select features, label-encode categoricals, return X_train, X_test ready for XGBoost."""
    available = [f for f in feature_list if f in train_df.columns]
    missing = [f for f in feature_list if f not in train_df.columns]
    if missing:
        print(f'  Missing from data: {missing}')
    print(f'  Available features: {len(available)} / {len(feature_list)}')

    X_train = train_df[available].copy()
    X_test = test_df[available].copy() if test_df is not None else None

    # Encode categoricals (object or bool)
    for col in available:
        if X_train[col].dtype == 'object' or X_train[col].dtype == 'bool':
            cats = pd.Categorical(X_train[col])
            X_train[col] = cats.codes.astype('float64')
            if X_test is not None:
                X_test[col] = pd.Categorical(X_test[col], categories=cats.categories).codes.astype('float64')
            # Replace -1 (unseen categories) with 0
            X_train[col] = X_train[col].replace(-1, 0)
            if X_test is not None:
                X_test[col] = X_test[col].replace(-1, 0)

    return X_train, X_test, available

# XGBoost parameters — simpler than v4 for the smaller dataset
XGB_PARAMS = {
    'objective': 'reg:tweedie',
    'tweedie_variance_power': 1.5,
    'max_depth': 4,          # v4 used 6 — shallower for smaller data
    'eta': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.5,
    'min_child_weight': 20,  # v4 used 10 — higher for smaller data
    'reg_alpha': 1.0,         # v4 used 0.5 — more regularization
    'reg_lambda': 3.0,        # v4 used 2.0 — more regularization
    'seed': 42,
    'n_jobs': -1,
    'verbosity': 0,
}

N_FOLDS = 5
NUM_BOOST_ROUND = 200

models = {}
cv_scores = {}
feature_importances = {}
test_predictions = {}
oof_predictions = {}  # out-of-fold predictions for blending

for line in ['motor', 'buildings', 'contents']:
    print(f'\n{"="*60}')
    print(f'=== {line.upper()} XGBoost (k-fold CV) ===')
    print(f'{"="*60}')

    train_df = train[line]
    test_df = test[line]

    X_all, X_test, feat_cols = prepare_features(train_df, test_df, feature_selection[line])
    y_all = train_df['ultimate_claims'].values
    exposure_all = train_df['exposure'].values
    exp_test = test_df['exposure'].values

    # k-fold CV
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    fold_scores = []
    oof_pred = np.zeros(len(train_df))
    fold_importances = np.zeros(len(feat_cols))

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_all)):
        X_tr = X_all.iloc[tr_idx]
        X_val = X_all.iloc[val_idx]
        y_tr = y_all[tr_idx]
        y_val = y_all[val_idx]
        exp_tr = exposure_all[tr_idx]
        exp_val = exposure_all[val_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr, base_margin=np.log(exp_tr + 1e-10), weight=exp_tr)
        dval = xgb.DMatrix(X_val, label=y_val, base_margin=np.log(exp_val + 1e-10), weight=exp_val)

        model = xgb.train(XGB_PARAMS, dtrain, num_boost_round=NUM_BOOST_ROUND,
                          evals=[(dval, 'val')], early_stopping_rounds=30, verbose_eval=False)

        val_pred = np.maximum(model.predict(dval), 0)
        d2 = tweedie_d2(y_val, val_pred, p=1.5)
        fold_scores.append(d2)
        oof_pred[val_idx] = val_pred

        # Accumulate feature importance
        scores = model.get_score(importance_type='gain')
        for i, f in enumerate(feat_cols):
            fold_importances[i] += scores.get(f, 0)

        print(f'  Fold {fold+1}: D2 = {d2:.4f}, pred mean = {val_pred.mean():.1f} (actual {y_val.mean():.1f})')

    cv_scores[line] = np.mean(fold_scores)
    oof_predictions[line] = oof_pred
    print(f'  CV mean D2: {cv_scores[line]:.4f} (+/- {np.std(fold_scores):.4f})')
    print(f'  OOF pred mean: {oof_pred.mean():.4f} (actual mean: {y_all.mean():.4f})')

    # Feature importance (averaged across folds)
    imp = pd.DataFrame({'feature': feat_cols, 'importance': fold_importances / N_FOLDS})
    imp = imp.sort_values('importance', ascending=False)
    feature_importances[line] = imp
    print(f'  Top 10 features:')
    print(f'  {imp.head(10).to_string(index=False)}')

    # Train final model on all data for test predictions
    dtrain_full = xgb.DMatrix(X_all, label=y_all, base_margin=np.log(exposure_all + 1e-10), weight=exposure_all)
    dtest = xgb.DMatrix(X_test, base_margin=np.log(exp_test + 1e-10))
    final_model = xgb.train(XGB_PARAMS, dtrain_full, num_boost_round=NUM_BOOST_ROUND, verbose_eval=False)
    models[line] = final_model

    test_pred = np.maximum(final_model.predict(dtest), 0.01)
    test_predictions[line] = test_pred
    print(f'  Test predictions: mean={test_pred.mean():.4f}, min={test_pred.min():.4f}, max={test_pred.max():.4f}')

print(f'\n{"="*60}')
print(f'=== Summary: CV D2 Scores ===')
for line in ['motor', 'buildings', 'contents']:
    print(f'  {line}: {cv_scores[line]:.4f}')
print(f'  Average: {np.mean(list(cv_scores.values())):.4f}')

In [0]:
# Step 5: Generate submission file
SUB_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/submissions_emerging_v1/'
os.makedirs(SUB_DIR, exist_ok=True)

submission = sample_submission.copy()

for line in ['motor', 'buildings', 'contents']:
    test_df = test[line]
    pred = test_predictions[line] * TREND_FACTORS[line]  # apply trend uplift for 2025
    pred = np.clip(pred, 0.01, 50000)
    pred_map = dict(zip(test_df['row_id'], pred))
    mask = submission['row_id'].isin(pred_map)
    submission.loc[mask, 'prediction'] = submission.loc[mask, 'row_id'].map(pred_map)
    print(f'{line}: {mask.sum()} predictions added (trend factor={TREND_FACTORS[line]})')

submission['prediction'] = submission['prediction'].clip(lower=0.01, upper=50000)
submission['prediction'] = submission['prediction'].replace([np.inf, -np.inf], 50000).fillna(0.01)
sub_path = os.path.join(SUB_DIR, 'xgboost_tweedie_v1.csv')
submission.to_csv(sub_path, index=False)
print(f'\nSaved: {sub_path} ({len(submission)} rows)')
print(f'Prediction stats: mean={submission["prediction"].mean():.4f}, min={submission["prediction"].min():.4f}, max={submission["prediction"].max():.4f}')

In [0]:
# Step 6: Save CV scores, feature importance, and OOF predictions for Notebook 3 (GLM & Blend)
EXCEL_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/Excel files_emerging_v1/'
os.makedirs(EXCEL_DIR, exist_ok=True)

# Excel checkpoint
with pd.ExcelWriter(os.path.join(EXCEL_DIR, 'xgboost_v1_results.xlsx'), engine='openpyxl') as writer:
    scores_df = pd.DataFrame([
        {'line': line, 'd2_cv': cv_scores[line]} for line in ['motor', 'buildings', 'contents']
    ])
    scores_df.to_excel(writer, sheet_name='D2 Scores', index=False)
    for line in ['motor', 'buildings', 'contents']:
        feature_importances[line].to_excel(writer, sheet_name=f'{line} importance', index=False)
print(f'Saved: {EXCEL_DIR}xgboost_v1_results.xlsx')

# Save OOF predictions and feature importance for GLM & Blend notebook
import pickle
save_data = {
    'oof_predictions': oof_predictions,
    'feature_importances': feature_importances,
    'cv_scores': cv_scores,
    'test_predictions': test_predictions,
    'trend_factors': TREND_FACTORS,
}
save_path = os.path.join(PROC_DIR, 'xgboost_v1_results.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(save_data, f)
print(f'Saved: {save_path}')

print(f'\nValidation D2 scores:')
for line in ['motor', 'buildings', 'contents']:
    print(f'  {line}: {cv_scores[line]:.4f}')
print(f'  Average: {np.mean(list(cv_scores.values())):.4f}')

## Step 7: AI/Human Decision Log

### What AI did
- Loaded processed emerging data from Notebook 01
- Prepared test data with same cleaning + enrichment joins
- Data-driven feature selection (all features minus targets/IDs/PII/leakage)
- Fit XGBoost Tweedie (power=1.5) with 5-fold CV (adapted from v4's time-based split)
- Simpler model: max_depth=4 (vs 6), min_child_weight=20 (vs 10), reg_alpha=1.0 (vs 0.5)
- log(exposure) as base_margin offset, exposure as weights
- Applied Part 1 trend factors for 2025 test predictions
- Saved submission, Excel checkpoint, OOF predictions + feature importance for GLM & Blend

### Key adaptations vs v4
1. **k-fold CV** instead of time-based split (only 2 years available)
2. **Simpler model** — max_depth=4, min_child_weight=20, more regularization (5% of data)
3. **Data-driven feature selection** — no Features_v3.xlsx needed
4. **Trend factors from Part 1** — motor 1.04, buildings 1.096, contents 1.049